# Audio Feature Extraction — Spectrogram / CLAP / Wav2Vec2 / WavLM

| Feature | Wrapper / Function | Output |
|---|---|---|
| MFCC + Mel-spectrogram | `compute_mfcc` / `compute_melspec` | `(n_mfcc, T)` / `(n_mels, T)` |
| OpenSMILE eGeMAPSv02 | `OpensmileWrapper` | `(T, 25)` LLD features |
| CLAP audio embeddings | `ClapWrapper` | `(1, 512)` float32 |
| Wav2Vec2 features | `Wav2vec2Wrapper` | `(T, 768)` hidden states |
| WavLM features | `WavlmWrapper` | `list[Tensor (1, T, 768)]` per layer |

Demo uses `multispeaker.wav` (61 s, 16 kHz).

In [1]:
from pathlib import Path

audio_path = Path("../tests/fixtures/audio/multispeaker.wav")
print(f"Audio file: {audio_path}")
print(f"Exists: {audio_path.exists()}")

Audio file: ../tests/fixtures/audio/multispeaker.wav
Exists: True


## 1. Load Audio

In [2]:
from exordium.audio.io import load_audio

# load_audio returns a (torch.Tensor, int) — waveform at 16 kHz, mono
waveform, sr = load_audio(audio_path, target_sample_rate=16000)

print(f"Waveform: shape={waveform.shape}, dtype={waveform.dtype}")
print(f"Sample rate: {sr} Hz")
print(f"Duration: {waveform.shape[-1] / sr:.2f} s")

Waveform: shape=torch.Size([979325]), dtype=torch.float32
Sample rate: 16000 Hz
Duration: 61.21 s


## 2. Spectrogram — MFCC and Mel-Spectrogram

In [3]:
from exordium.audio.spectrogram import compute_melspec, compute_mfcc

mfcc, mfcc_preemph = compute_mfcc(waveform, sr)
print(f"MFCC:          shape={mfcc.shape}  (n_mfcc x time)")
print(f"MFCC preemph:  shape={mfcc_preemph.shape}")

melspec_db, melspec_db_preemph = compute_melspec(waveform, sr)
print(f"Mel-spec:          shape={melspec_db.shape}  (n_mels x time)")
print(f"Mel-spec preemph:  shape={melspec_db_preemph.shape}")

MFCC:          shape=(40, 6121)  (n_mfcc x time)
MFCC preemph:  shape=(40, 6121)
Mel-spec:          shape=(80, 6121)  (n_mels x time)
Mel-spec preemph:  shape=(80, 6121)


## 3. OpenSMILE Features

In [4]:
from exordium.audio.smile import OpensmileWrapper

# Process directly from the audio file path — no need to load separately
smile = OpensmileWrapper(feature_set="egemaps", feature_level="lld")
smile_features = smile(audio_path, sr=sr)

print(f"OpenSMILE eGeMAPSv02 LLD: shape={smile_features.shape}, dtype={smile_features.dtype}")
print(f"  rows = frames,  cols = {smile_features.shape[1]} low-level descriptors")

OpenSMILE eGeMAPSv02 LLD: shape=(6116, 25), dtype=float32
  rows = frames,  cols = 25 low-level descriptors


## 4. CLAP Embeddings

In [5]:
from exordium.audio.clap import CLAP_SAMPLE_RATE, ClapWrapper
from exordium.audio.io import load_audio

# CLAP requires 48 kHz audio
waveform_48k, _ = load_audio(audio_path, target_sample_rate=CLAP_SAMPLE_RATE)

clap = ClapWrapper(device_id=-1)  # -1 → CPU
clap_emb = clap(waveform_48k, sample_rate=CLAP_SAMPLE_RATE)

print(f"CLAP embedding: shape={clap_emb.shape}, dtype={clap_emb.dtype}")
print("  (B=1, 512-dim audio embedding aligned with text)")

/Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
objc[28762]: Class AVFFrameReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/libavdevice.61.3.100.dylib (0x106bec7a0) and /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x16c4a43a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[28762]: Class AVFAudioReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/libavdevice.61.3.100.dylib (0x106bec7f0) and /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x16c4a43f8). This may cause spurious casting failures and mysterious crashes. One of

CLAP embedding: shape=torch.Size([1, 512]), dtype=torch.float32
  (B=1, 512-dim audio embedding aligned with text)


## 5. Wav2Vec2 Features

In [6]:
from exordium.audio.wav2vec2 import Wav2vec2Wrapper

# expects 16 kHz mono waveform — reuse waveform loaded in cell 1
wav2vec2 = Wav2vec2Wrapper(device_id=-1)  # -1 → CPU
wav2vec2_features = wav2vec2(waveform)  # input: Tensor (T,)

print(f"Wav2Vec2 features: shape={wav2vec2_features.shape}")
print("  (T_frames x 768 hidden-dim)")

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 32197.83it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec2 features: shape=torch.Size([3060, 768])
  (T_frames x 768 hidden-dim)


## 6. WavLM Features

In [7]:
from exordium.audio.wavlm import WavlmWrapper

# expects 16 kHz mono waveform — reuse waveform from cell 1
wavlm = WavlmWrapper(device_id=-1, model_name="base+")  # -1 → CPU
wavlm_layers = wavlm(waveform)  # returns list[Tensor] — one per transformer layer

print(f"WavLM hidden layers: {len(wavlm_layers)}")
for i, layer in enumerate(wavlm_layers):
    print(f"  Layer {i:2d}: shape={tuple(layer.shape)}  (B x T_frames x hidden_dim)")

Loading weights: 100%|██████████| 248/248 [00:00<00:00, 87168.98it/s]


WavLM hidden layers: 12
  Layer  0: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  1: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  2: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  3: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  4: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  5: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  6: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  7: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  8: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer  9: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer 10: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
  Layer 11: shape=(1, 3060, 768)  (B x T_frames x hidden_dim)
